# PUNTO B) - Predicción de Abandono en Estudiantes Terciarios

**Instituto Superior de Formación Técnica N° 197**  
**Alumno:** Lucas Nahuel Llanes  
**Materia:** Procesamiento de Aprendizaje Automático  
**Docente:** Scudero, Yanina Ximena  
**Carrera:** Tecnicatura Superior en Ciencia de Datos e Inteligencia Artificial  
**Año:** 2026

---

Este notebook reproduce el flujo del script `AD.py`: carga los Excel, prepara los datos, entrena un árbol de decisión, evalúa, interpreta y predice sobre `DatosFinal`.

## 1. Importación de librerías y carga de datos

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

def buscar_excel(palabra):
    encontrados = [f for f in glob.glob('*.xlsx')
                   if palabra.lower() in f.lower()
                   and not f.startswith('predicciones')
                   and not f.startswith('~$')]
    if not encontrados:
        raise FileNotFoundError(f"No hay .xlsx con '{palabra}'")
    print(f'Usando: {encontrados[0]}')
    return encontrados[0]

ARCHIVO_ENTRENAMIENTO = buscar_excel('Entrenamiento')
ARCHIVO_NUEVOS = buscar_excel('DatosFinal')
SEMILLA = 42

## 2. Preparación de los datos

In [ ]:
NOMBRES = {
    'edad': 'edad', 'genero': 'genero', 'carrera': 'carrera',
    'PromedioPrimerCuatrimestre': 'promedio',
    'CantMateriasAprobadasPrimerCuatrimestre': 'aprobadas',
    'CantMateriasDesaprobadasPrimerCuatrimestre': 'desaprobadas',
    'AsistenciaPromedio(%)': 'asistencia',
    'trabaja/NoTrabaja': 'trabaja',
    'DistanciaDomicilioAlInstituto(Kms)': 'distancia_km',
    'ActividadesExtracurriculares(Estudio)': 'extracurriculares',
    'EstadoFinal': 'estado_final',
}

def preparar(df):
    df = df.rename(columns=NOMBRES).copy()
    for col in ['genero', 'carrera', 'trabaja', 'extracurriculares']:
        df[col] = df[col].astype(str).str.strip()
    df['carrera'] = (df['carrera']
                     .str.replace('TÉCNICATURA', 'TECNICATURA')
                     .str.replace('TECNICATURA SUPERIOR EN ', ''))
    df['genero'] = df['genero'].map({'m': 0, 'f': 1})
    df['trabaja'] = df['trabaja'].map({'No': 0, 'Si': 1})
    df['extracurriculares'] = df['extracurriculares'].map({'No': 0, 'Si': 1})
    if 'estado_final' in df.columns:
        df['estado_final'] = df['estado_final'].map({'Abandonó': 1, 'Continúa': 0})
    return df

datos = preparar(pd.read_excel(ARCHIVO_ENTRENAMIENTO, sheet_name=0))
nuevos = preparar(pd.read_excel(ARCHIVO_NUEVOS, sheet_name=0))

carreras = sorted(set(datos['carrera']) | set(nuevos['carrera']))
datos['carrera'] = pd.Categorical(datos['carrera'], categories=carreras)
nuevos['carrera'] = pd.Categorical(nuevos['carrera'], categories=carreras)
datos = pd.get_dummies(datos, columns=['carrera'], prefix='carr', dtype=int)
nuevos = pd.get_dummies(nuevos, columns=['carrera'], prefix='carr', dtype=int)

assert datos.isna().sum().sum() == 0
assert nuevos.isna().sum().sum() == 0

X = datos.drop(columns='estado_final')
y = datos['estado_final']
X_nuevos = nuevos[X.columns]

print(f'Filas de entrenamiento: {len(datos)} | A predecir: {len(nuevos)}')
print(y.value_counts().to_string())

## 3. División train / test y entrenamiento

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEMILLA, stratify=y)

modelo = DecisionTreeClassifier(
    criterion='gini', max_depth=4, min_samples_leaf=5, random_state=SEMILLA)
modelo.fit(X_train, y_train)

## 4. Evaluación

In [ ]:
pred_train = modelo.predict(X_train)
pred_test = modelo.predict(X_test)

print(f'Accuracy entrenamiento: {accuracy_score(y_train, pred_train):.2f}')
print(f'Accuracy prueba:        {accuracy_score(y_test, pred_test):.2f}')

print(pd.DataFrame(confusion_matrix(y_test, pred_test, labels=[0, 1]),
                   index=['Real: Continúa', 'Real: Abandonó'],
                   columns=['Pred: Continúa', 'Pred: Abandonó']))

print(classification_report(y_test, pred_test,
                            target_names=['Continúa', 'Abandonó'], zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred_test, display_labels=['Continúa', 'Abandonó'], cmap='Blues')
plt.title('Matriz de confusión - conjunto de prueba')
plt.tight_layout(); plt.show()

## 5. Interpretación: importancia de variables y reglas

In [ ]:
importancias = (pd.Series(modelo.feature_importances_, index=X.columns)
                .sort_values(ascending=False))
usadas = importancias[importancias > 0]
print(usadas.round(3).to_string())

usadas.sort_values().plot.barh(color='#4C72B0')
plt.xlabel('Importancia (reducción de Gini)')
plt.title('Importancia de variables')
plt.tight_layout(); plt.show()

print(export_text(modelo, feature_names=list(X.columns)))

plt.figure(figsize=(22, 10))
plot_tree(modelo, feature_names=X.columns, class_names=['Continúa', 'Abandonó'],
          filled=True, rounded=True, fontsize=9)
plt.tight_layout(); plt.show()

## 6. Estabilidad de las variables importantes

In [ ]:
acc_test = accuracy_score(y_test, pred_test)
acc_trivial = max(y_test.mean(), 1 - y_test.mean())
raiz = X.columns[modelo.tree_.feature[0]]

top_por_division = []
for semilla in range(5):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.3,
                                      random_state=semilla, stratify=y)
    arbol = DecisionTreeClassifier(max_depth=4, min_samples_leaf=5,
                                   random_state=SEMILLA).fit(Xa, ya)
    top = (pd.Series(arbol.feature_importances_, index=X.columns)
           .sort_values(ascending=False).index[:3].tolist())
    top_por_division.append(top[0])
    print(f'División {semilla}: accuracy prueba = {accuracy_score(yb, arbol.predict(Xb)):.2f} | '
          f'top 3 = {", ".join(top)}')

print(f'\nRaíz: {raiz} | Accuracy test: {acc_test:.2f} | Trivial: {acc_trivial:.2f}')
print(f'Variable principal por división: {top_por_division}')

## 7. Predicción sobre DatosFinal

In [ ]:
resultado = pd.read_excel(ARCHIVO_NUEVOS, sheet_name=0)
resultado['Prediccion'] = ['Abandonó' if p == 1 else 'Continúa'
                           for p in modelo.predict(X_nuevos)]
resultado['ProbAbandono'] = modelo.predict_proba(X_nuevos)[:, 1].round(2)
resultado = resultado.sort_values('ProbAbandono', ascending=False)
resultado.to_excel('predicciones_DatosFinal.xlsx', index=False)

print(resultado['Prediccion'].value_counts().to_string())
resultado.head(10)

## 8. Conclusiones

El árbol de decisión permite detectar que **el dataset no tiene señal predictiva**: el accuracy en prueba (0.50) es menor que el de un modelo trivial (0.53) y las variables "importantes" cambian según la división. Se recomienda: sistema de alerta por inasistencias, modalidad virtual para estudiantes que viven lejos, tutorías focalizadas y becas para quienes trabajan.